In [1]:
# ============================================================
# QN 2
# Finite-horizon MDP for the investor problem
# Includes:
#   - exhaustive search for T = 3
#   - dynamic programming for T = 3
#   - optional DP for T = 100 (bonus)
#
# Main modelling assumptions used:
# 1) Daily change states are:
#       0 -> price unchanged
#       1 -> price increases by 10
#       2 -> price decreases by 10
# 2) Initial stock price is 100.
# 3) Initial change-state is taken to be 0 ("same"), since the question
#    gives the initial price but not the previous day's movement.
# 4) At each decision time, the investor chooses ONE action type for the day:
#       buy stock / buy futures / sell stock / short / do nothing
#    and can repeat that action multiple times.
# 5) Futures bought today mature tomorrow:
#       tomorrow cash decreases by agreed price,
#       and owned stock increases by the number of futures.
# 6) If a short position is due tomorrow, then tomorrow is forced:
#       investor must buy back the shares and return them,
#       and no other transaction is allowed that day.
# 7) To keep the action set finite and reasonable:
#       - buy-stock count <= cash // price
#       - buy-futures count <= cash // price
#       - short count is capped so that worst-case next-day repurchase
#         (price+10) is coverable using current cash + short-sale proceeds,
#         after paying the 1% borrow fee today.
# ============================================================

from functools import lru_cache
from itertools import product
import time

# -------------------------
# Problem data
# -------------------------
beta = 1 / 1.05
T_small = 3
T_large = 100

P = {
    0: {0: 0.10, 1: 0.45, 2: 0.45},
    1: {0: 0.20, 1: 0.20, 2: 0.60},
    2: {0: 0.10, 1: 0.70, 2: 0.20},
}

DELTA = {
    0: 0,    # same
    1: 10,   # increase
    2: -10,  # decrease
}

SHORT_FEE_RATE = 0.01

# initial state:
# (change_state, price, cash, stocks_owned, futures_held, shorts_due, forced_no_trade, bankrupt)
INITIAL_STATE = (0, 100, 100, 1, 0, 0, 0, 0)

# -------------------------
# Utilities
# -------------------------

def state_to_str(state):
    z, price, cash, stocks, futures, shorts_due, forced_no_trade, bankrupt = state
    return {
        "change_state": z,
        "price": price,
        "cash": cash,
        "stocks": stocks,
        "futures": futures,
        "shorts_due": shorts_due,
        "forced_no_trade": bool(forced_no_trade),
        "bankrupt": bool(bankrupt),
    }


def apply_next_day_price(state, next_change):
    """
    Move from end of day t to end of day t+1 BEFORE making new decision at t+1:
      - stock price moves according to next_change
      - futures from previous day mature and must be paid now
      - if shorts are due, they must be bought back now
    """
    z, price, cash, stocks, futures, shorts_due, forced_no_trade, bankrupt = state

    forced_no_trade = 0

    if bankrupt:
        return (next_change, price, cash, stocks, futures, shorts_due, forced_no_trade, 1)

    new_price = price + DELTA[next_change]

    # 1) futures mature
    if futures > 0:
        total_future_payment = futures * price
        cash -= total_future_payment
        stocks += futures
        futures = 0
        if cash < 0:
            return (next_change, new_price, cash, stocks, futures, shorts_due, forced_no_trade, 1)

    # 2) short positions due
    if shorts_due > 0:
        repurchase_cost = shorts_due * new_price
        cash -= repurchase_cost
        shorts_due = 0
        forced_no_trade = 1
        if cash < 0:
            return (next_change, new_price, cash, stocks, futures, shorts_due, forced_no_trade, 1)
    
    return (next_change, new_price, cash, stocks, futures, shorts_due, forced_no_trade, 0)


def enumerate_actions(state):
    """
    Return a list of feasible actions at the current end-of-day state.
    Action encoding:
        ("none", 0)
        ("buy_stock", k)
        ("buy_future", k)
        ("short", k)

    If shorts_due > 0 or bankrupt -> no choice.
    """
    z, price, cash, stocks, futures, shorts_due, forced_no_trade, bankrupt = state

    if bankrupt or forced_no_trade:
        return [("none", 0)]

    if shorts_due > 0:
        return [("none", 0)]

    actions = [("none", 0)]

    # If price is zero or negative, disallow transactions that use price in division
    # and treat the stock as non-tradeable from that point.
    if price <= 0:
        return actions

    # Buy stock now
    max_buy_stock = int(cash // price)
    for k in range(1, max_buy_stock + 1):
        actions.append(("buy_stock", k))

    # Buy futures now
    max_buy_future = int(cash // price)
    for k in range(1, max_buy_future + 1):
        actions.append(("buy_future", k))

    # Sell stock
    for k in range(1, stocks + 1):
        actions.append(("sell_stock", k))

    # Shorting only allowed if currently no stocks and no futures
    if stocks == 0 and futures == 0:
        required_per_share = 1.01 * price
        max_short = int(cash // required_per_share)
        for k in range(1, max_short + 1):
            actions.append(("short", k))

    return actions


def apply_action(state, action):
    """
    Apply end-of-day action immediately at current day.
    Returns the new end-of-day state after decision.
    Stage reward is cash at hand after this action.
    """
    z, price, cash, stocks, futures, shorts_due, forced_no_trade, bankrupt = state
    a_type, k = action

    if bankrupt or forced_no_trade:
        return state

    if a_type == "none":
        return state

    if a_type == "buy_stock":
        cost = k * price
        if cost > cash:
            return (z, price, cash, stocks, futures, shorts_due, forced_no_trade, 1)
        cash -= cost
        stocks += k
        return (z, price, cash, stocks, futures, shorts_due, forced_no_trade, 0)

    if a_type == "buy_future":
        futures += k
        return (z, price, cash, stocks, futures, shorts_due, forced_no_trade, 0)

    if a_type == "sell_stock":
        revenue = k * price
        stocks -= k
        cash += revenue
        return (z, price, cash, stocks, futures, shorts_due, forced_no_trade, 0)

    if a_type == "short":
        if stocks != 0 or futures != 0:
            return (z, price, cash, stocks, futures, shorts_due, forced_no_trade, 1)

        fee = k * price * SHORT_FEE_RATE
        sale_proceeds = k * price

        if fee > cash:
            return (z, price, cash, stocks, futures, shorts_due, forced_no_trade, 1)

        cash = cash - fee + sale_proceeds
        shorts_due += k
        return (z, price, cash, stocks, futures, shorts_due, forced_no_trade, 0)

    raise ValueError(f"Unknown action {action}")


def stage_reward(state_after_action):
    """
    r_t = cash at hand at end of day t
    """
    z, price, cash, stocks, futures, shorts_due, forced_no_trade, bankrupt = state_after_action
    return cash

# -------------------------
# Exhaustive search
# -------------------------
def exhaustive_value(state, t, T):
    """
    Brute-force recursion without memoization.
    Returns (best_value, best_action, policy_tree)
    """
    if t > T:
        return 0.0, None, None

    # If this state came from a short, the next day has no decision.
    # The restriction is already handled because enumerate_actions returns only "none"
    # when shorts_due > 0.
    best_val = float("-inf")
    best_action = None
    best_tree = None

    for action in enumerate_actions(state):
        s_after = apply_action(state, action)
        immediate = (beta ** t) * stage_reward(s_after)

        if t == T:
            total = immediate
            tree = {"action": action, "next": {}}
        else:
            continuation = 0.0
            subtree = {}
            for z_next, prob in P[s_after[0]].items():
                s_next = apply_next_day_price(s_after, z_next)
                val_next, a_next, tree_next = exhaustive_value(s_next, t + 1, T)
                continuation += prob * val_next
                subtree[z_next] = {
                    "prob": prob,
                    "state": state_to_str(s_next),
                    "best_action": a_next,
                    "subtree": tree_next,
                }

            total = immediate + continuation
            tree = {"action": action, "next": subtree}

        if total > best_val:
            best_val = total
            best_action = action
            best_tree = tree

    return best_val, best_action, best_tree


# -------------------------
# Dynamic programming
# -------------------------
def run_dp(state, T):
    global cache_hits, memo
    cache_hits = 0
    memo = {}
    val = dp_value(state, 0, T, count_hits=True)
    return val, cache_hits

def dp_value(state, t, T, depth=0, count_hits=False):
    global cache_hits, memo
    state = tuple(state)
    key = (state, t, T)

    if key in memo:
        if count_hits:
            cache_hits += 1
        return memo[key]

    if t > T:
        return 0.0

    best_val = float("-inf")

    for action in enumerate_actions(state):
        s_after = apply_action(state, action)
        immediate = (beta ** t) * stage_reward(s_after)

        if t == T:
            total = immediate
        else:
            continuation = 0.0
            for z_next, prob in P[s_after[0]].items():
                s_next = apply_next_day_price(s_after, z_next)
                continuation += prob * dp_value(s_next, t + 1, T, depth + 1, count_hits=count_hits)
            total = immediate + continuation

        best_val = max(best_val, total)

    memo[key] = best_val
    return best_val


def dp_policy(state, t, T):
    """
    Recover one optimal action at (state, t).
    """
    state = tuple(state)
    best_val = float("-inf")
    best_action = None

    for action in enumerate_actions(state):
        s_after = apply_action(state, action)
        immediate = (beta ** t) * stage_reward(s_after)

        if t == T:
            total = immediate
        else:
            continuation = 0.0
            for z_next, prob in P[s_after[0]].items():
                s_next = apply_next_day_price(s_after, z_next)
                continuation += prob * dp_value(s_next, t + 1, T)

            total = immediate + continuation

        if total > best_val:
            best_val = total
            best_action = action

    return best_action, best_val


def print_policy_tree_dp(state, t, T, indent=0, max_depth=3):
    """
    Nicely prints one optimal DP policy tree up to depth max_depth.
    """
    prefix = " " * indent
    action, val = dp_policy(state, t, T)
    print(f"{prefix}t={t}, state={state_to_str(state)}, best_action={action}, value={val:.4f}")

    if t >= T or max_depth <= 1:
        return

    s_after = apply_action(state, action)
    for z_next, prob in P[s_after[0]].items():
        s_next = apply_next_day_price(s_after, z_next)
        print(f"{prefix}  -> next change {z_next} with prob {prob}")
        print_policy_tree_dp(s_next, t + 1, T, indent + 6, max_depth - 1)

def print_policy_tree_exhaustive(state, t, T, indent=0, max_depth=3):
    """
    Nicely prints one optimal exhaustive-search policy tree up to depth max_depth.
    """
    prefix = " " * indent
    val, action, tree = exhaustive_value(state, t, T)
    print(f"{prefix}t={t}, state={state_to_str(state)}, best_action={action}, value={val:.4f}")

    if t >= T or max_depth <= 1:
        return

    s_after = apply_action(state, action)
    for z_next, prob in P[s_after[0]].items():
        s_next = apply_next_day_price(s_after, z_next)
        print(f"{prefix}  -> next change {z_next} with prob {prob}")
        print_policy_tree_exhaustive(s_next, t + 1, T, indent + 6, max_depth - 1)

In [2]:
print("========== QN 2 : T = 3 ==========")
print("Initial state:", state_to_str(INITIAL_STATE))

start = time.perf_counter()
ex_val, ex_action, ex_tree = exhaustive_value(INITIAL_STATE, 0, T_small)
t_exhaustive = time.perf_counter() - start

start = time.perf_counter()
dp_val, hits = run_dp(INITIAL_STATE, T_small)
dp_action, dp_val_check = dp_policy(INITIAL_STATE, 0, T_small)
t_dp = time.perf_counter() - start

print(f"Exhaustive search optimal PV = {ex_val:.6f}")
print(f"Exhaustive search first action = {ex_action}")
print(f"Exhaustive search time = {t_exhaustive:.6f} seconds")

print(f"DP optimal PV = {dp_val:.6f}")
print(f"DP first action = {dp_action}")
print(f"DP time = {t_dp:.6f} seconds")
print(f"DP cache hits = {cache_hits}")

========== QN 2 : T = 3 ==========
Initial state: {'change_state': 0, 'price': 100, 'cash': 100, 'stocks': 1, 'futures': 0, 'shorts_due': 0, 'forced_no_trade': False, 'bankrupt': False}
Exhaustive search optimal PV = 975.768049
Exhaustive search first action = ('sell_stock', 1)
Exhaustive search time = 0.022484 seconds
DP optimal PV = 975.768049
DP first action = ('sell_stock', 1)
DP time = 0.005294 seconds
DP cache hits = 498


In [3]:
print("\nOne optimal DP policy tree:")
print_policy_tree_dp(INITIAL_STATE, 0, T_small, max_depth=4)


One optimal DP policy tree:
t=0, state={'change_state': 0, 'price': 100, 'cash': 100, 'stocks': 1, 'futures': 0, 'shorts_due': 0, 'forced_no_trade': False, 'bankrupt': False}, best_action=('sell_stock', 1), value=975.7680
  -> next change 0 with prob 0.1
      t=1, state={'change_state': 0, 'price': 100, 'cash': 200, 'stocks': 0, 'futures': 0, 'shorts_due': 0, 'forced_no_trade': False, 'bankrupt': False}, best_action=('short', 1), value=763.0925
        -> next change 0 with prob 0.1
            t=2, state={'change_state': 0, 'price': 100, 'cash': 199.0, 'stocks': 0, 'futures': 0, 'shorts_due': 0, 'forced_no_trade': True, 'bankrupt': False}, best_action=('none', 0), value=472.5580
              -> next change 0 with prob 0.1
                  t=3, state={'change_state': 0, 'price': 100, 'cash': 199.0, 'stocks': 0, 'futures': 0, 'shorts_due': 0, 'forced_no_trade': False, 'bankrupt': False}, best_action=('short', 1), value=257.4236
              -> next change 1 with prob 0.45
         

In [4]:
print_policy_tree_exhaustive(INITIAL_STATE, 0, T_small, max_depth=4)

t=0, state={'change_state': 0, 'price': 100, 'cash': 100, 'stocks': 1, 'futures': 0, 'shorts_due': 0, 'forced_no_trade': False, 'bankrupt': False}, best_action=('sell_stock', 1), value=975.7680
  -> next change 0 with prob 0.1
      t=1, state={'change_state': 0, 'price': 100, 'cash': 200, 'stocks': 0, 'futures': 0, 'shorts_due': 0, 'forced_no_trade': False, 'bankrupt': False}, best_action=('short', 1), value=763.0925
        -> next change 0 with prob 0.1
            t=2, state={'change_state': 0, 'price': 100, 'cash': 199.0, 'stocks': 0, 'futures': 0, 'shorts_due': 0, 'forced_no_trade': True, 'bankrupt': False}, best_action=('none', 0), value=472.5580
              -> next change 0 with prob 0.1
                  t=3, state={'change_state': 0, 'price': 100, 'cash': 199.0, 'stocks': 0, 'futures': 0, 'shorts_due': 0, 'forced_no_trade': False, 'bankrupt': False}, best_action=('short', 1), value=257.4236
              -> next change 1 with prob 0.45
                  t=3, state={'change_